# Lecture 5 Evaluation Exercise: M/G/3 Case Study (Generator)

This notebook is for the **instructor**. It generates the synthetic log used in the Lecture 5 evaluation exercise: an $M/G/3$ queue with Poisson arrivals and two-phase (Erlang) service times with a fixed setup overhead.

- True model: $M/G/3$ with $\lambda = 0.78$ and service $S = s_0 + 	ext{Erlang}(k=2,\, \mu=1.2)$ where $s_0 = 0.4$ time units.
- The datasets written to `data/lecture5_mg3_case_study.csv` and `notebooks/data/lecture5_mg3_case_study.csv` are the files students analyse.

In [ ]:
import math
import os
from typing import Dict, List

import numpy as np
import pandas as pd


def generate_mg3_erlang_dataset(T: float,
                                lam: float,
                                s0: float,
                                mu: float,
                                seed: int = 123,
                                max_servers: int = 3,
                                erlang_shape: int = 2) -> pd.DataFrame:
    '''Generate per-job dataset for an M/G/3 queue with Erlang-two service.'''
    rng = np.random.default_rng(seed)
    t = 0.0
    next_arrival = t + rng.exponential(1.0 / lam)

    servers: List[Dict[str, float]] = []
    queue: List[Dict[str, float]] = []
    rows: List[Dict[str, float]] = []

    def start_service(now: float, job: Dict[str, float]) -> None:
        service_tail = rng.gamma(erlang_shape, 1.0 / mu)
        s = s0 + service_tail
        job['service_time'] = float(s)
        job['start_service_time'] = float(now)
        job['completion_time'] = float(now + s)
        servers.append(job)

    while True:
        next_completion = min((job['completion_time'] for job in servers), default=math.inf)
        t_next = min(next_arrival, next_completion)
        if t_next == math.inf:
            break
        t = t_next

        if next_arrival <= next_completion:
            if t <= T:
                q_len = len(queue) + len(servers)
                job = {
                    'arrival_time': float(t),
                    'queue_len_at_arrival': int(q_len),
                }
                if len(servers) < max_servers:
                    start_service(t, job)
                else:
                    queue.append(job)
            next_arrival = t + rng.exponential(1.0 / lam)
        else:
            idx = min(range(len(servers)), key=lambda i: servers[i]['completion_time'])
            job = servers.pop(idx)
            if job['arrival_time'] <= T:
                completion_time = job['completion_time']
                rows.append({
                    'arrival_time': job['arrival_time'],
                    'queue_len_at_arrival': job['queue_len_at_arrival'],
                    'service_time': job['service_time'],
                    'start_service_time': job['start_service_time'],
                    'completion_time': completion_time,
                    'wait_time': max(0.0, job['start_service_time'] - job['arrival_time']),
                    'system_time': completion_time - job['arrival_time'],
                })
            if queue:
                next_job = queue.pop(0)
                start_service(t, next_job)

        if next_arrival > T and not servers and not queue and t >= T:
            break

    return pd.DataFrame(
        rows,
        columns=[
            'arrival_time',
            'queue_len_at_arrival',
            'service_time',
            'start_service_time',
            'completion_time',
            'wait_time',
            'system_time',
        ],
    )


LAMBDA_TRUE = 0.78
S0_TRUE = 0.4
MU_TRUE = 1.2
ERLANG_SHAPE = 2
T = 4000.0

DATA_PATHS = [
    'data/lecture5_mg3_case_study.csv',
    'notebooks/data/lecture5_mg3_case_study.csv',
]

os.makedirs('data', exist_ok=True)
os.makedirs('notebooks/data', exist_ok=True)

df = generate_mg3_erlang_dataset(
    T=T,
    lam=LAMBDA_TRUE,
    s0=S0_TRUE,
    mu=MU_TRUE,
    seed=2025,
    max_servers=3,
    erlang_shape=ERLANG_SHAPE,
)
for data_path in DATA_PATHS:
    df.to_csv(data_path, index=False)
    print(f"Generated dataset: {data_path} with {len(df)} rows.")

T_obs = float(df['arrival_time'].max())
lam_hat = len(df) / T_obs
mean_service = float(df['service_time'].mean())
servers = 3
rho_hat = lam_hat * mean_service / servers
print(f"Rough check: lambda_hat={lam_hat:.4f}, m1_hat={mean_service:.4f}, rho_hat={rho_hat:.4f}")

## Instructor Solution Outline

This section gives a worked solution for the evaluation exercise, using the main tools from Lectures 2–4 and commenting explicitly on *why* each step is taken.

- **Arrival process (Poisson model):**
  We estimate the arrival rate $\hat\lambda$ from counts on $[0,T]$ and form an asymptotic CI using the Poisson variance $\Var(\hat\lambda) pprox \lambda/T$. We then check modelling assumptions with inter-arrival histograms, log-survival plots, and rate-by-subinterval diagnostics to see whether a homogeneous Poisson process is reasonable.
- **Service-time model (setup + Erlang tail):**
  We treat service times as a deterministic setup $s_0$ plus an Erlang-$2$ tail, $S = s_0 + 	ext{Erlang}(k=2, \mu)$, which captures a fixed overhead followed by two exponential processing stages. We estimate $s_0$ from the minimum observed service time and fit the Erlang rate $\mu$ via method-of-moments, then compare the model to the empirical distribution via histogram, ECDF, and tail diagnostics.
- **Performance metrics ($W_q$, $L_q$, utilisation):**
  Using the fitted model and three servers, we compute utilisation $\hat
ho = \hat\lambda \hat m_1 / 3$, empirical mean waiting time $\overline W_q$, and the mean number in queue $L_q$ from arrivals (using `queue_len_at_arrival`) and via Little's Law $L_q pprox \hat\lambda\,\overline W_q`. This checks internal consistency of the model and data.
- **Simulation-based uncertainty and comparison:**
  We simulate the fitted $M/G/3$ model repeatedly, compute the run-level averages of $W_q$ and $L_q$, and use the CLT over replications to obtain 95% confidence intervals: $\hat m \pm 1.96\,\hat s/\sqrt{B}$. The final figure overlays these simulation-based distributions (a parametric bootstrap under the fitted model) and CIs with the empirical estimates to visually assess how well the fitted model explains the observed performance and where it might miss.

In [ ]:
import math
from typing import Dict, List

import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")

Z_95 = 1.959963984540054

# 1. Arrival process diagnostics
T_obs = float(df['arrival_time'].max())
n_jobs = len(df)
lambda_hat = n_jobs / T_obs
se_lambda = math.sqrt(max(lambda_hat, 1e-12) / T_obs)
lam_lo = lambda_hat - Z_95 * se_lambda
lam_hi = lambda_hat + Z_95 * se_lambda
print(f"Arrival rate: lambda_hat={lambda_hat:.4f} (95% CI [{lam_lo:.4f}, {lam_hi:.4f}])")

arrival_times = np.sort(df['arrival_time'].to_numpy())
inter_arrivals = np.diff(arrival_times)
mean_ia = inter_arrivals.mean()
rate_from_ia = 1.0 / max(mean_ia, 1e-12)

fig, axes = plt.subplots(2, 2, figsize=(10, 6))

ax = axes[0, 0]
bins = np.linspace(0, np.quantile(inter_arrivals, 0.99), 40)
ax.hist(inter_arrivals, bins=bins, density=True, alpha=0.6, label='Empirical')
grid = np.linspace(0, bins[-1], 200)
ax.plot(grid, rate_from_ia * np.exp(-rate_from_ia * grid), 'r--', label='Exp fit')
ax.set_title('Inter-arrivals vs Exp tail')
ax.set_xlabel('Inter-arrival time')
ax.set_ylabel('Density')
ax.legend()

ax = axes[0, 1]
sorted_ia = np.sort(inter_arrivals)
surv = 1.0 - np.arange(1, len(sorted_ia) + 1) / (len(sorted_ia) + 1)
ax.plot(sorted_ia, surv, label='Empirical')
ax.plot(grid, np.exp(-rate_from_ia * grid), 'r--', label='Exp fit')
ax.set_yscale('log')
ax.set_title('Inter-arrival log-survival')
ax.set_xlabel('Inter-arrival time')
ax.set_ylabel('P(A > t)')
ax.legend()

ax = axes[1, 0]
K = 10
edges = np.linspace(0.0, T_obs, K + 1)
counts, _ = np.histogram(arrival_times, bins=edges)
centres = 0.5 * (edges[:-1] + edges[1:])
ax.bar(centres, counts / (T_obs / K), width=T_obs / K * 0.8)
ax.axhline(lambda_hat, color='r', linestyle='--', label='Overall rate')
ax.set_title('Estimated arrival rate by sub-interval')
ax.set_xlabel('Time')
ax.set_ylabel('Rate in bin')
ax.legend()

ax = axes[1, 1]
ax.step(arrival_times, np.arange(1, len(arrival_times) + 1), where='post', label='N(t)')
ax.plot([0, T_obs], [0, lambda_hat * T_obs], 'r--', label='lambda_hat t')
ax.set_title('Cumulative arrivals vs linear trend')
ax.set_xlabel('t')
ax.set_ylabel('N(t)')
ax.legend()

plt.tight_layout()
plt.show()

# 2. Service-time model: setup + Erlang-2 tail
S = df['service_time'].to_numpy()
s0_hat = float(S.min())
Y = np.clip(S - s0_hat, 1e-12, None)
shape = 2.0
mean_tail = float(Y.mean())
std_tail = float(Y.std(ddof=1))
se_tail_mean = std_tail / math.sqrt(max(len(Y), 1))
mu_hat = shape / mean_tail
se_mu = (shape / (mean_tail ** 2)) * se_tail_mean
mu_lo = mu_hat - Z_95 * se_mu
mu_hi = mu_hat + Z_95 * se_mu
m1_model = s0_hat + shape / mu_hat
print(f"Service model: S = s0 + Erlang(k=2, mu) with s0_hat={s0_hat:.4f}")
print(f"  mu_hat={mu_hat:.4f} (95% CI [{mu_lo:.4f}, {mu_hi:.4f}]),  m1_model={m1_model:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
ax = axes[0]
bins_S = np.linspace(S.min(), np.quantile(S, 0.99), 40)
ax.hist(S, bins=bins_S, density=True, alpha=0.6, label='Empirical')
grid_S = np.linspace(bins_S[0], bins_S[-1], 300)

def erlang_pdf(x: np.ndarray, mu: float, shape: float = 2.0) -> np.ndarray:
    tail = np.maximum(x, 0.0)
    return (mu ** shape) * (tail ** (shape - 1)) * np.exp(-mu * tail) / math.gamma(shape)

model_pdf = np.where(grid_S >= s0_hat, erlang_pdf(grid_S - s0_hat, mu_hat), 0.0)
ax.plot(grid_S, model_pdf, 'r--', label='s0 + Erlang')
ax.set_title('Service-time histogram')
ax.set_xlabel('service_time')
ax.set_ylabel('Density')
ax.legend()

ax = axes[1]
S_sorted = np.sort(S)
ecdf = np.arange(1, len(S_sorted) + 1) / (len(S_sorted) + 1)
ax.step(S_sorted, ecdf, where='post', label='ECDF')
model_cdf = np.where(
    grid_S < s0_hat,
    0.0,
    1.0 - np.exp(-mu_hat * (grid_S - s0_hat)) * (1.0 + mu_hat * (grid_S - s0_hat))
)
ax.plot(grid_S, model_cdf, 'r--', label='Model CDF')
ax.set_title('Service ECDF vs model')
ax.set_xlabel('service_time')
ax.set_ylabel('F_S(t)')
ax.legend()

ax = axes[2]
tail_grid = np.linspace(0, np.quantile(Y, 0.995), 200)
surv_emp = 1.0 - np.arange(1, len(Y) + 1) / (len(Y) + 1)
ax.plot(np.sort(Y), surv_emp, label='Empirical tail')
surv_model = np.exp(-mu_hat * tail_grid) * (1.0 + mu_hat * tail_grid)
ax.plot(tail_grid, surv_model, 'r--', label='Erlang-2 tail')
ax.set_yscale('log')
ax.set_title('Service tail log-survival')
ax.set_xlabel('S - s0_hat')
ax.set_ylabel('P(S - s0_hat > x)')
ax.legend()

plt.tight_layout()
plt.show()

# 3. Utilisation and empirical performance
servers = 3
rho_model = lambda_hat * m1_model / servers
Wq_emp = float(df['wait_time'].mean())
Lq_emp_arrivals = float(np.maximum(df['queue_len_at_arrival'].to_numpy() - servers, 0).mean())
Lq_ll_emp = lambda_hat * Wq_emp
zero_wait_frac = float((df['wait_time'] == 0.0).mean())
Wq_positive = float(df.loc[df['wait_time'] > 0.0, 'wait_time'].mean()) if (df['wait_time'] > 0.0).any() else 0.0
print(f"Fraction of zero waits = {zero_wait_frac:.3f},  mean wait | Wq>0 = {Wq_positive:.4f}")
print(f"Utilisation (model-based): rho_hat={rho_model:.4f}")
print(f"Empirical mean wait Wq_emp={Wq_emp:.4f}")
print(f"Empirical Lq from arrivals: Lq_emp={Lq_emp_arrivals:.4f}")
print(f"Little's Law estimate: Lq_LL=lambda_hat * Wq_emp={Lq_ll_emp:.4f}")

# 4. Simulation-based Wq and Lq for fitted model

def generate_mg3_offset_erlang_dataset(T: float,
                                       lam: float,
                                       s0: float,
                                       mu: float,
                                       seed: int,
                                       max_servers: int = 3,
                                       erlang_shape: int = 2) -> pd.DataFrame:
    '''Simulate the fitted M/G/3 model with Erlang-2 service.'''
    rng = np.random.default_rng(seed)
    t = 0.0
    next_arrival = t + rng.exponential(1.0 / lam)
    servers_active: List[Dict[str, float]] = []
    queue: List[Dict[str, float]] = []
    rows: List[Dict[str, float]] = []

    def start_service(now: float, job: Dict[str, float]) -> None:
        service_tail = rng.gamma(erlang_shape, 1.0 / mu)
        s = s0 + service_tail
        job['service_time'] = float(s)
        job['start_service_time'] = float(now)
        job['completion_time'] = float(now + s)
        servers_active.append(job)

    while True:
        next_completion = min((job['completion_time'] for job in servers_active), default=math.inf)
        t_next = min(next_arrival, next_completion)
        if t_next == math.inf:
            break
        t = t_next

        if next_arrival <= next_completion:
            if t <= T:
                q_len = len(queue) + len(servers_active)
                job = {
                    'arrival_time': float(t),
                    'queue_len_at_arrival': int(q_len),
                }
                if len(servers_active) < max_servers:
                    start_service(t, job)
                else:
                    queue.append(job)
            next_arrival = t + rng.exponential(1.0 / lam)
        else:
            idx = min(range(len(servers_active)), key=lambda i: servers_active[i]['completion_time'])
            job = servers_active.pop(idx)
            if job['arrival_time'] <= T:
                completion_time = job['completion_time']
                rows.append({
                    'arrival_time': job['arrival_time'],
                    'queue_len_at_arrival': job['queue_len_at_arrival'],
                    'service_time': job['service_time'],
                    'start_service_time': job['start_service_time'],
                    'completion_time': completion_time,
                    'wait_time': max(0.0, job['start_service_time'] - job['arrival_time']),
                    'system_time': completion_time - job['arrival_time'],
                })
            if queue:
                next_job = queue.pop(0)
                start_service(t, next_job)

        if next_arrival > T and not servers_active and not queue and t >= T:
            break

    return pd.DataFrame(rows)


B = 200
rng = np.random.default_rng(8)
Wq_samples = []
Lq_samples = []
for b in range(B):
    sim_df = generate_mg3_offset_erlang_dataset(
        T=T_obs,
        lam=lambda_hat,
        s0=s0_hat,
        mu=mu_hat,
        seed=int(rng.integers(0, 1_000_000)),
        max_servers=servers,
        erlang_shape=2,
    )
    if len(sim_df) == 0:
        continue
    Wq_samples.append(float(sim_df['wait_time'].mean()))
    Lq_samples.append(float(np.maximum(sim_df['queue_len_at_arrival'].to_numpy() - servers, 0).mean()))

Wq_samples = np.array(Wq_samples)
Lq_samples = np.array(Lq_samples)

for name, samples, empirical in [
    ('W_q', Wq_samples, Wq_emp),
    ('L_q', Lq_samples, Lq_emp_arrivals),
]:
    mean_est = float(samples.mean())
    se_est = float(samples.std(ddof=1) / math.sqrt(len(samples)))
    ci_lo = mean_est - Z_95 * se_est
    ci_hi = mean_est + Z_95 * se_est
    print(f"Simulation {name}: mean={mean_est:.4f} (95% CI [{ci_lo:.4f}, {ci_hi:.4f}]), empirical={empirical:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(Wq_samples, bins=25, alpha=0.7)
axes[0].axvline(Wq_emp, color='r', linestyle='--', label='Empirical')
axes[0].set_title('Simulation Wq means')
axes[0].set_xlabel('Wq')
axes[0].legend()
axes[1].hist(Lq_samples, bins=25, alpha=0.7)
axes[1].axvline(Lq_emp_arrivals, color='r', linestyle='--', label='Empirical')
axes[1].set_title('Simulation Lq means')
axes[1].set_xlabel('Lq')
axes[1].legend()
plt.tight_layout()
plt.show()

## Reading the Results: A Short Investigation

To turn the raw numbers into insight, it helps to walk through the outputs step by step and ask: *does this make sense for this system?*

1. **Empirical picture from the log**
   - Observation horizon is about $T_	ext{obs}pprox 4{,}000$, with an estimated arrival rate $\hat\lambda$ around $0.78$ jobs per time unit.
   - Most jobs start immediately ($\mathbb P(W_q=0)$ is above 85%), yet the conditional mean wait among the remaining jobs is comparable to a service time, giving a small but noticeable mean wait overall.
   - Queue lengths rarely exceed the three in-service jobs, consistent with a three-server setting operating below critical load.
2. **Service-time structure**
   - There is a visible lower bound (fixed setup) followed by a curved log-tail, matching an Erlang-2 (two exponential phases) distribution rather than a memoryless exponential.
   - The fitted $s_0$ and $\mu$ imply a first moment near 2 time units, which combined with three servers yields utilisation well below 1.
3. **Model-vs-data comparison**
   - Simulation under the fitted $M/G/3$ parameters produces distributions for $W_q$ and $L_q$ that bracket the empirical estimates, lending credibility to the queue model.
   - Deviations would suggest either time-varying arrivals, heavier service tails, or model misspecification (e.g. priority rules), which should be discussed in student solutions.

## Interpretation and Limitations

- **Arrival diagnostics:** If the inter-arrival histogram and log-survival closely match an exponential and the rate-by-subinterval plot fluctuates around $\hat\lambda$ without trend, a homogeneous Poisson process is plausible. Strong diurnal effects or heavy deviations from the exponential tail would suggest a non-homogeneous model.
- **Service diagnostics:** An Erlang-2 tail is justified when there is a visible lower bound plus a *curved* log-survival that eventually straightens on a linear scale (two exponential stages). If the tail is clearly heavier or lighter, a different family (Gamma with different shape, lognormal, etc.) might be required.
- **Performance metrics:** Consistency between $\hat
ho$, empirical $\overline W_q$, and $L_q$ (via both arrivals and Little's Law) should be emphasised. For three servers operating below critical load, even small modelling errors can produce noticeable differences in $W_q$, so uncertainty quantification is important.
- **Uncertainty:** The simulation-based CIs above treat the fitted model as the truth. Students should combine these with input uncertainty (e.g. bootstrap over arrival/service samples) when arguing about robustness.
- **Finite-horizon effects:** All estimates come from a single contiguous window. Early or late transients can bias $W_q$ and $L_q$; discarding a warm-up or using regenerative cycles would reduce this risk.

## LLM Grading Prompt (for Instructor Use)

You are an expert teaching assistant for a French M2 Data Science course on stochastic modelling and simulation.
You grade student solutions to the Lecture 5 evaluation exercise in `lecture5_evaluation_exercise.ipynb`,
which analyses the dataset `data/lecture5_mg3_case_study.csv` generated by this notebook.

### 🚨 The "Hidden" Truth (Instructor Eyes Only)
The dataset was actually generated by an **M/G/3** system, despite the exercise prompt suggesting an **M/G/2** model.
- **The "Easter Egg"**: A careful student might notice that some arrivals find 2 jobs in the system (`queue_len_at_arrival == 2`) but have `wait_time == 0`. In an M/G/2 system, this is impossible (they would have to wait).
- **Grading Implication**:
    - **Detects Violation**: Students who detect this and conclude the system has >2 servers should receive **top marks (19–20)**, provided the rest of their work is rigorous and follow all the important points.
    - **Follows Prompt**: Students who proceed with the M/G/2 analysis (as instructed) **should NOT be penalized**. They can still earn up to 17/20 if their M/G/2 analysis is really perfect.

### Context
- **Course**: Poisson processes, M/M/1, M/G/1, basic error control (Lectures 2–4).
- **Key Concept**: The service times have a **fixed overhead** . Detecting this is crucial.
- **Tools**: Poisson estimation, plug-in estimators, parametric fitting (Shifted Exp/Erlang), uncertainty quantification (CLT, Bootstrap, Simulation).

### Grading Scale (0–20)
- **0–5**: Fail. Major tasks missing or code doesn't run.
- **6–9**: Weak. Major conceptual errors (e.g., treating system as M/M/1, ignoring $s_0$).
- **10–13**: Pass. Core tasks done. Parameters estimated roughly.
- **14–16**: Strong. Good model (Shifted Exp/Erlang), good diagnostics (log-survival), clear CIs.
- **17–18**: Very Strong. Rigorous simulation comparison, bootstrap CIs, excellent communication.
- **19–20**: Exceptional. Detects the M/G/3 "Easter Egg" OR provides a flawless, deeply critical M/G/2 analysis with advanced uncertainty quantification.

### Criteria (total 20 points)
Score the submission on the following axes, then sum and round to the nearest half-point if helpful:

1. **Modelling and Assumptions (0–4)**
   - 0: No coherent model.
   - 3: Reasonable M/G/2 model with stated assumptions.
   - 4: Critical discussion of assumptions. **Bonus**: Identifies the structural violation (3rd server) in the data.

2. **Arrival Process Analysis and $\hat\lambda$ (0–4)**
   - 0: No meaningful analysis.
   - 3: Correct $\hat\lambda$ with CI and basic plots.
   - 4: Rigorous check of Homogeneous Poisson assumption (log-survival, sub-intervals, etc.).

3. **Service-Time Modelling (0–4)**
   - **Critical**: Must identify the fixed overhead ($s_0$). If they model as pure Exponential (no shift), max score is 1/4.
   - 3: Fits Shifted Exponential or Shifted Erlang/Gamma.
   - 4: Validates tail fit (log-survival) and discusses parameter uncertainty (CIs for $\mu$).

4. **Performance Metrics: $W_q$, $L_q$, and Utilisation (0–4)**
   - 0: No meaningful analysis.
   - 3: Correct empirical $\overline{W}_q, L_q, \hat\rho$.
   - 4: Compares empirical values to **Simulation** or **Approximation** predictions to assess model fit.

5. **Uncertainty, Diagnostics, and Communication (0–4)**
   - 0: No uncertainty quantification.
   - 3: Basic CIs provided for parameters.
   - 4: **High-Level**: Uses Bootstrap on empirical data OR Simulation CIs to rigorously compare model vs reality. Notebook is professional and clear.

### Instructions to the LLM Grader
- **Check for $s_0$**: This is a binary check for competence. No $s_0$ = significant penalty.
- **Check for Easter Egg**: Look for checks on `queue_len_at_arrival` vs `wait_time`.
- **Be Nuanced**: A Shifted Exponential fit is "correct enough" (full points). They don't need to guess the exact Erlang-2 generator, just a model that fits the data well.
- **Feedback**: Briefly justify the score with 3–5 sentences, highlighting strengths (e.g., "Great use of bootstrap") and areas for improvement.